# Stage 50 — Risk-adjusted metrics & divergence flags

**Owner:** Risk Analyst  
**Pipeline stage:** `pipelines.stage_50_metrics`

Compute per-borrower / segment / portfolio risk-adjusted metrics (CoV, RAROC, Sortino family, diversification ratio) and the RAROC-vs-Sortino divergence early-warning flags. Segment roll-ups use block-sum loss covariance, never an average of per-borrower ratios.

**Reads:** `persons_scored`, `corr_matrix`  
**Writes:** `per_borrower_metrics`, `segment_metrics_*`, `divergence_flags`, `portfolio_metrics`

> This notebook is *modular*: it only needs its upstream artifacts to exist in the
> shared store. You can re-run just this stage without touching the others.


## 1. Setup

In [ ]:
# --- setup: make the project root importable + open the shared artifact store ---
import sys, os
ROOT = os.path.abspath("..")          # notebooks/ lives one level below the project root
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from pipelines import ArtifactStore
store = ArtifactStore("output/etl")   # the SAME store every stage reads/writes
print("artifact store:", store.root)
print("existing artifacts:", store.list())


## 2. Run this stage

In [ ]:
from pipelines.stage_50_metrics import run as run_metrics

result = run_metrics(store, copula_type="clayton", lgd=0.45,
                     segment_cols=["city_name", "risk_archetype"])
print(result.summary())


## 3. Inspect what this stage produced

In [ ]:
display(store.read_df("per_borrower_metrics").head())
display(store.read_df("segment_metrics_city_name"))
print("portfolio_metrics:", store.read_json("portfolio_metrics"))
flags = store.read_df("divergence_flags")
print("RAROC-vs-Sortino divergence flags:", len(flags))
display(flags.head())
